<a href="https://colab.research.google.com/github/NataliiaZubenia/NMR_prediction/blob/main/LabAnalyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install deps (first run only)
!pip install pyyaml pandas openpyxl

import yaml, pandas as pd, numpy as np

# Load YAML thresholds
with open("thresholds.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

def classify_numeric(val, limits):
    """
    Map a numeric value to category using YAML limits:
    {normal: x, watch: y, alert: z, critical: w}
    """
    if val is None or pd.isna(val):
        return "NA"
    if "critical" in limits and val >= limits["critical"]: return "Critical"
    if "alert"    in limits and val >= limits["alert"]:    return "Alert"
    if "watch"    in limits and val >= limits["watch"]:    return "Watch"
    return "Normal"


In [ ]:
#  VISCOSITY
FILE = "viscosity.csv"   # your file
PROFILE = "engine"       # or "hydraulic" / "transmission"

df = pd.read_csv(FILE)

cand_visc_cols = [c for c in df.columns if str(c).lower() in ("visc_cst","visc100_cst","visc40_cst","viscosity")]
if not cand_visc_cols:
    raise ValueError("No viscosity column found. Expected Visc_cSt / Visc100_cSt / Visc40_cSt / Viscosity")
visc_col = cand_visc_cols[0]

if "Nominal_cSt" not in df.columns:
    print("No 'Nominal_cSt' column. Δ% cannot be computed → all 'Normal'. Add column to enable rule.")
    df["ViscDeltaPct"] = np.nan
else:
    df["ViscDeltaPct"] = (df[visc_col] - df["Nominal_cSt"]) / df["Nominal_cSt"] * 100.0

thr = CFG["profiles"][PROFILE]["methods"]["VISC"]["viscosity_delta_pct"]

def visc_status(delta, profile):
    if pd.isna(delta):
        return "Normal"
    ad = abs(delta)

    # Custom rules
    if profile == "hydraulic":
        if ad > 7:
            return "Alert"
    elif profile == "engine":
        if ad > 15 or delta < 10:
            return "Alert"

    #  Default YAML thresholds
    if "critical" in thr and ad >= thr["critical"]: return "Critical"
    if "alert"    in thr and ad >= thr["alert"]:    return "Alert"
    if "watch"    in thr and ad >= thr["watch"]:    return "Watch"
    return "Normal"

df["Viscosity_Status"] = df["ViscDeltaPct"].apply(lambda d: visc_status(d, PROFILE))

display(df[["SampleID", visc_col, "Nominal_cSt", "ViscDeltaPct", "Viscosity_Status"]])
df.to_excel("viscosity_evaluated.xlsx", index=False)
print(" Wrote viscosity_evaluated.xlsx")


In [ ]:
# FTIR
FILE = "ftir.csv"
PROFILE = "engine"   # typical

df = pd.read_csv(FILE)

ft_cfg = CFG["profiles"][PROFILE]["methods"]["FTIR"]
use_allow = ft_cfg.get("use_allowable_if_present", True)
base_thr  = ft_cfg["thresholds"]

# Map parameter -> possible Allowable column names
allow_map = {
    "Fuel_pct": "Allowable_Fuel_pct",
    "Water_pct":"Allowable_Water_pct",
    "Oxidation_index":"Allowable_Oxidation_index",
    "Nitration_index":"Allowable_Nitration_index",
    "Sulfation_index":"Allowable_Sulfation_index",
    "Soot_index":"Allowable_Soot_index",
}

params = [p for p in base_thr.keys() if p in df.columns]
if not params:
    raise ValueError("No FTIR parameters found. Expected columns like Fuel_pct, Water_pct, Oxidation_index...")

def thr_for_row(row, param):
    """Row-level thresholds: use per-sample Allowable if present, else YAML defaults."""
    if use_allow:
        a_col = allow_map.get(param)
        if a_col and a_col in row and pd.notna(row[a_col]) and row[a_col] != 0:
            a = float(row[a_col])
            return {"watch": 0.8*a, "alert": a, "critical": 1.5*a}
    return base_thr[param]

def classify_param(val, limits):
    if pd.isna(val): return "NA"
    if "critical" in limits and val >= limits["critical"]: return "Critical"
    if "alert"    in limits and val >= limits["alert"]:    return "Alert"
    if "watch"    in limits and val >= limits["watch"]:    return "Watch"
    return "Normal"

# Evaluate each parameter
for p in params:
    df[f"{p}_Status"] = [
        classify_param(v, thr_for_row(row, p))
        for v, (_, row) in zip(df[p].values, df.iterrows())
    ]

# Minimal single-row rules that only depend on FTIR values (engine rules needing viscosity/ICP are skipped here)
# Example: thermal_stress rule using only Oxidation/Nitration + Fuel
def thermal_stress_row(row):
    ox = row.get("Oxidation_index", np.nan)
    ni = row.get("Nitration_index", np.nan)
    fuel = row.get("Fuel_pct", np.nan)
    ox_thr = thr_for_row(row, "Oxidation_index")
    ni_thr = thr_for_row(row, "Nitration_index")
    fu_thr = thr_for_row(row, "Fuel_pct")
    cond = ((not pd.isna(ox) and ox >= ox_thr["alert"]) or
            (not pd.isna(ni) and ni >= ni_thr["alert"])) and \
           (pd.isna(fuel) or fuel < fu_thr["watch"])
    return "Triggered" if cond else "OK"

df["Rule_ThermalStress"] = df.apply(thermal_stress_row, axis=1)

# Save / Show
keep_cols = ["SampleID"] + params + [f"{p}_Status" for p in params] + ["Rule_ThermalStress"]
display(df[keep_cols])
df.to_excel("ftir_evaluated.xlsx", index=False)
print("Wrote ftir_evaluated.xlsx")


In [ ]:
# ICP-OES
FILE = "icp.csv"
PROFILE = "engine"   # or "hydraulic" / "transmission" for different limits

df = pd.read_csv(FILE)

# Optional: rename columns from instrument headers to canonical element symbols
rename_map = {
    # "Iron(ppm)": "Fe", "Copper(ppm)": "Cu", ...
}
df = df.rename(columns=rename_map)

icp_thr = CFG["profiles"][PROFILE]["methods"]["ICP"]["thresholds_mgkg"]

# Which elements from YAML exist in the CSV?
elems = [e for e in icp_thr.keys() if e in df.columns]
if not elems:
    raise ValueError("No ICP elements matched. Ensure columns like Fe, Cu, Si... are present.")

for e in elems:
    df[f"{e}_Status"] = df[e].apply(lambda v: classify_numeric(v, icp_thr[e]))

# Save / Show
keep = ["SampleID"] + elems + [f"{e}_Status" for e in elems]
display(df[keep])
df.to_excel("icp_evaluated.xlsx", index=False)
print("Wrote icp_evaluated.xlsx")


In [ ]:
# LNF / ISO 4406
FILE = "lnf.csv"
PROFILE = "hydraulic"  # LNF targets usually apply to hydraulics/transmissions

df = pd.read_csv(FILE)

lnf_cfg = CFG["profiles"][PROFILE]["methods"]["LNF"]
target_str = lnf_cfg["iso_target"]          # e.g. "18/16/13"
tol = int(lnf_cfg.get("class_tolerance", 2))

def parse_iso_code(x):
    """Accept '18/16/13' or three separate columns; return tuple of ints (c4,c6,c14)."""
    if isinstance(x, str) and "/" in x:
        parts = x.split("/")
        return tuple(int(p) for p in parts[:3])
    return None

# Build actual code per row
if "ISO_code" in df.columns:
    df[["ISO_4","ISO_6","ISO_14"]] = df["ISO_code"].apply(
        lambda s: pd.Series(parse_iso_code(s))
    )

if not set(["ISO_4","ISO_6","ISO_14"]).issubset(df.columns):
    raise ValueError("Provide either ISO_code ('18/16/13') or columns ISO_4, ISO_6, ISO_14")

t4,t6,t14 = [int(x) for x in target_str.split("/")[:3]]

def lnf_status(row):
    c4,c6,c14 = int(row["ISO_4"]), int(row["ISO_6"]), int(row["ISO_14"])
    over = (c4 - t4, c6 - t6, c14 - t14)
    max_over = max(over)
    if max_over >= tol:
        return "Alert"
    elif max_over >= 1:
        return "Watch"
    else:
        return "Normal"

df["LNF_Status"] = df.apply(lnf_status, axis=1)

# Save / Show
keep = ["SampleID","ISO_4","ISO_6","ISO_14","LNF_Status"]
if "ISO_code" in df.columns: keep = ["SampleID","ISO_code","ISO_4","ISO_6","ISO_14","LNF_Status"]
display(df[keep])
df.to_excel("lnf_evaluated.xlsx", index=False)
print(" Wrote lnf_evaluated.xlsx")
